In [1]:
import numpy as np
import scipy.io as sio
import os
import matplotlib.pyplot as plt
from pathlib import Path

class BoldSimulationLoader:
    """
    Class to handle BOLD timeseries simulation data from MATLAB files
    """
    
    def __init__(self, data_dir):
        """
        Initialize with directory containing sim*.mat files
        
        Args:
            data_dir (str): Path to directory containing simulation files
        """
        self.data_dir = Path(data_dir)
        self.sim_files = sorted(list(self.data_dir.glob('sim*.mat')))
        print(f"Found {len(self.sim_files)} simulation files")
    
    def load_simulation(self, sim_number):
        """
        Load a specific simulation file
        
        Args:
            sim_number (int): Simulation number (1-28)
            
        Returns:
            dict: Dictionary containing simulation data
        """
        sim_file = self.data_dir / f'sim{sim_number}.mat'
        
        if not sim_file.exists():
            raise FileNotFoundError(f"Simulation file {sim_file} not found")
        
        # Load MATLAB file
        mat_data = sio.loadmat(sim_file)
        
        # Extract variables (handle both possible naming conventions)
        data = {}
        for key in ['Nsubjects', 'Ntimepoints', 'Nnodes', 'ts', 'net']:
            if key in mat_data:
                data[key] = mat_data[key]
            elif key.lower() in mat_data:
                data[key] = mat_data[key.lower()]
        
        # Convert to more convenient format
        n_subjects = int(data['Nsubjects'].flatten()[0])
        n_timepoints = int(data['Ntimepoints'].flatten()[0])
        n_nodes = int(data['Nnodes'].flatten()[0])
        
        return {
            'n_subjects': n_subjects,
            'n_timepoints': n_timepoints,
            'n_nodes': n_nodes,
            'timeseries_concat': data['ts'],  # All subjects concatenated
            'ground_truth_networks': data['net']  # Subject x Nodes x Nodes
        }
    
    def extract_subject_timeseries(self, sim_data, subject_idx):
        """
        Extract timeseries for a specific subject
        
        Args:
            sim_data (dict): Data from load_simulation()
            subject_idx (int): Subject index (0-based)
            
        Returns:
            np.ndarray: Timeseries for subject (timepoints x nodes)
        """
        n_timepoints = sim_data['n_timepoints']
        n_subjects = sim_data['n_subjects']
        
        if subject_idx >= n_subjects:
            raise ValueError(f"Subject index {subject_idx} >= n_subjects {n_subjects}")
        
        start_idx = subject_idx * n_timepoints
        end_idx = start_idx + n_timepoints
        
        return sim_data['timeseries_concat'][start_idx:end_idx, :]
    
    def extract_subject_network(self, sim_data, subject_idx):
        """
        Extract ground truth network for a specific subject
        
        Args:
            sim_data (dict): Data from load_simulation()
            subject_idx (int): Subject index (0-based)
            
        Returns:
            np.ndarray: Ground truth network matrix (nodes x nodes)
        """
        return sim_data['ground_truth_networks'][subject_idx, :, :]
    
    def process_all_subjects(self, sim_data, analysis_func):
        """
        Apply analysis function to all subjects in a simulation
        
        Args:
            sim_data (dict): Data from load_simulation()
            analysis_func (callable): Function that takes timeseries and returns result
            
        Returns:
            list: Results for each subject
        """
        results = []
        
        for subject_idx in range(sim_data['n_subjects']):
            ts = self.extract_subject_timeseries(sim_data, subject_idx)
            result = analysis_func(ts)
            results.append(result)
            
        return results
    
    def summary_stats(self, sim_data):
        """
        Print summary statistics for a simulation
        """
        print(f"Simulation Summary:")
        print(f"  Subjects: {sim_data['n_subjects']}")
        print(f"  Timepoints per subject: {sim_data['n_timepoints']}")
        print(f"  Network nodes: {sim_data['n_nodes']}")
        print(f"  Total timeseries shape: {sim_data['timeseries_concat'].shape}")
        print(f"  Ground truth networks shape: {sim_data['ground_truth_networks'].shape}")


In [2]:
import numpy as np
import networkx as nx

def unroll_dag_for_time_window(static_net: np.ndarray, k: int) -> nx.DiGraph:
    """
    Unrolls a static causal graph into a time-expanded DiGraph over a window k.

    This function builds a graph based on two key hypotheses:
    1.  Self-Markov Hypothesis: Every node i at time t-1 causes itself at time t.
        (Edge: i_{t-1} -> i_{t})
    2.  Lag 1 Causal Hypothesis: A static link i -> j implies a causal effect
        from node i at time t-1 to node j at time t.
        (Edge: i_{t-1} -> j_{t})

    Args:
        static_net: An (N x N) numpy array representing the static graph,
                    where static_net[i, j] > 0 indicates a causal link from i to j.
        k: The number of past time steps to unroll (e.g., k=3 unrolls back to t-3).

    Returns:
        A NetworkX DiGraph object representing the full, unrolled causal structure.
        Nodes are named in the format 'nodeID_t-lag'.
    """
    if k < 1:
        raise ValueError("Time window k must be at least 1.")

    N_nodes = static_net.shape[0]
    G = nx.DiGraph()

    # The main loop iterates through each time step transition, from the past to the present.
    # We are creating edges that connect slice (t-lag) to slice (t-lag+1).
    for lag in range(k, 0, -1):  # Iterates from k down to 1
        source_time_slice = lag      # e.g., t-3
        target_time_slice = lag - 1  # e.g., t-2

        # Loop through all nodes to add edges for this time transition
        for i in range(N_nodes):
            source_node_i = f"{i}_t-{source_time_slice}"

            # Hypothesis 2: Self-Markov (auto-causal) links
            # Every node i at t-lag causes itself at t-(lag-1)
            target_node_i = f"{i}_t-{target_time_slice}"
            G.add_edge(source_node_i, target_node_i, type='auto')

            # Hypothesis 1: Lag 1 Causal (cross-causal) links
            # Check for causal effects originating from node i
            for j in range(N_nodes):
                # If the static graph has a link i -> j...
                if i != j and static_net[i, j] > 0:
                    # ...create an edge from i at t-lag to j at t-(lag-1)
                    target_node_j = f"{j}_t-{target_time_slice}"
                    weight = static_net[i, j]
                    G.add_edge(source_node_i, target_node_j, weight=weight, type='cross')
    
    # Ensure nodes in the final time slice (t-0) are created, even if they have no incoming edges.
    # Note: add_edge already does this, but this is an explicit safeguard.
    for i in range(N_nodes):
        G.add_node(f"{i}_t-0")
        
    return G

In [4]:
import networkx as nx 


loader = BoldSimulationLoader("./raw/")
observations = {}
dags = {}
for sim_num in range(1, len(loader.sim_files) + 1): 
    sim_data = loader.load_simulation(sim_num)
    ts_concatenated = sim_data['timeseries_concat'].shape
    points_per_patient = sim_data['n_timepoints']
    num_patients = ts_concatenated[0] // points_per_patient
    size_single_ts = ts_concatenated[0] // num_patients
    assert num_patients == sim_data['n_subjects'], "Number of patients does not match number of subjects in data"
    ts_single_patients = {}
    for i in range(num_patients):
        from_ = points_per_patient*i 
        to_ = points_per_patient*(i+1)
        ts_single_patients[i] = sim_data['timeseries_concat'][from_:to_]
        assert ts_single_patients[i].shape == (size_single_ts, sim_data['n_nodes']), "Single patient time series shape mismatch"
    
    for i in range(num_patients):
        # cut observations above 200 rows 
        if ts_single_patients[i].shape[0] > 200:
            ts_single_patients[i] = ts_single_patients[i][:200, :]
    
    print(f"Patient {i} of simulation {20 + sim_num} time series shape: {ts_single_patients[i].shape}")

    dags_single_patients = {}
    
    for i in range(num_patients):
        adj_matrix = sim_data['ground_truth_networks'][i] # They are all the same, value is different but links are not
        np.fill_diagonal(adj_matrix, 0)
        G = nx.from_numpy_array(adj_matrix, create_using=nx.DiGraph)
        G_unrolled = unroll_dag_for_time_window(adj_matrix, k=3)
        dags_single_patients[i] = G_unrolled

    # already existing synthetic generative processes are from 1 to 20
    # we will therefore create a dictionary with keys 21 to 21 + n_sim (28) = 49
    observations[20 + sim_num] = ts_single_patients
    dags[20 + sim_num] = dags_single_patients

Found 28 simulation files
Patient 49 of simulation 21 time series shape: (200, 5)
Patient 49 of simulation 22 time series shape: (200, 10)
Patient 49 of simulation 23 time series shape: (200, 15)
Patient 49 of simulation 24 time series shape: (200, 50)
Patient 49 of simulation 25 time series shape: (200, 5)
Patient 49 of simulation 26 time series shape: (200, 10)
Patient 49 of simulation 27 time series shape: (200, 5)
Patient 49 of simulation 28 time series shape: (200, 5)
Patient 49 of simulation 29 time series shape: (200, 5)
Patient 49 of simulation 30 time series shape: (200, 5)
Patient 49 of simulation 31 time series shape: (200, 10)
Patient 49 of simulation 32 time series shape: (200, 10)
Patient 49 of simulation 33 time series shape: (200, 5)
Patient 49 of simulation 34 time series shape: (200, 5)
Patient 49 of simulation 35 time series shape: (200, 5)
Patient 49 of simulation 36 time series shape: (200, 5)
Patient 49 of simulation 37 time series shape: (200, 10)
Patient 49 of s

The DAG is coherent with what we expected. We proceed now to shape the data in the exact same format as the purely synthetic data we already use in the paper. This way, we can simply inject this new (observation, dag) pairs and run the code similarly. <br>
The way we do it is a dictionary of dictionaries. The first key is the generative process and the second key is the specific observation coming from the generative process <b>after changing the neighborhood randomly</b>.  <br>
The way this maps to the current situation is that we treat each simulation as a different generative process and the different patients as different observations of the same generative process. It is going to be the case that each patient will have different observations yet the same DAG. But the graph consistency between observations does not affect the process in any way.

TODO: how does the lenght and dimensionality of observation change wrt our synthetic data? 

In [5]:
for sim in observations.keys():
        print(f"Simulation {sim}, Patient {0}: {observations[sim][0].shape}")

observations_10_var = {}
dags_10_var = {}
observations_5_var = {}
dags_5_var = {}
for sim in observations.keys():
        if observations[sim][0].shape[1] == 5: 
                observations_5_var[sim] = observations[sim]
                dags_5_var[sim] = dags[sim]
        elif observations[sim][0].shape[1] == 10:
                observations_10_var[sim] = observations[sim]
                dags_10_var[sim] = dags[sim]
        else:  
                print(f"Unexpected number of nodes in simulation {sim}: {observations[sim][0].shape[1]}")


Simulation 21, Patient 0: (200, 5)
Simulation 22, Patient 0: (200, 10)
Simulation 23, Patient 0: (200, 15)
Simulation 24, Patient 0: (200, 50)
Simulation 25, Patient 0: (200, 5)
Simulation 26, Patient 0: (200, 10)
Simulation 27, Patient 0: (200, 5)
Simulation 28, Patient 0: (200, 5)
Simulation 29, Patient 0: (200, 5)
Simulation 30, Patient 0: (200, 5)
Simulation 31, Patient 0: (200, 10)
Simulation 32, Patient 0: (200, 10)
Simulation 33, Patient 0: (200, 5)
Simulation 34, Patient 0: (200, 5)
Simulation 35, Patient 0: (200, 5)
Simulation 36, Patient 0: (200, 5)
Simulation 37, Patient 0: (200, 10)
Simulation 38, Patient 0: (200, 5)
Simulation 39, Patient 0: (200, 5)
Simulation 40, Patient 0: (200, 5)
Simulation 41, Patient 0: (200, 5)
Simulation 42, Patient 0: (200, 5)
Simulation 43, Patient 0: (200, 5)
Simulation 44, Patient 0: (200, 5)
Simulation 45, Patient 0: (100, 5)
Simulation 46, Patient 0: (50, 5)
Simulation 47, Patient 0: (50, 5)
Simulation 48, Patient 0: (100, 5)
Unexpected numb

For the sake of dimensionality, we get rid of Simulation 23 and 24 because they have more than 10 variables 

In [6]:
import pickle 
with open('netsym_5.pkl', 'wb') as f:
    pickle.dump((observations_5_var, dags_5_var, None), f)

In [ ]:

with open('netsym_10.pkl', 'wb') as f:
    pickle.dump((observations_10_var, dags_10_var, None), f)